# 下载需求库

In [ ]:
%cd ~/work/
!pip install -q -r requirements.txt

# 解压mot20

In [ ]:
!cd ~/work/ && mkdir data && cd data && mkdir MOT20 && cd MOT20 && mkdir images && mkdir labels_with_ids && cd labels_with_ids && mkdir train
!unzip -q ~/data/data72344/MOT20.zip -d ~/work/data
!mv ~/work/data/MOT20/train/ ~/work/data/MOT20/images
!mv ~/work/data/MOT20/test/ ~/work/data/MOT20/images

# 生成标签

In [ ]:
%cd ~/work/src/
!python gen_labels_20.py

# 用hrnet_imagenet预训练模型训练

In [ ]:
%cd ~/work/src/
!python -W ignore train.py mot\
    --data_cfg ./lib/cfg/mot20.json\
    --K 500\
    --exp_id all_hrnet\
    --gpus 0\
    --batch_size 4\
    --arch 'hrnet_18'\
    --num_epochs 60

# 使用FairMOT提供的baseline模型在mot20上finetune

In [ ]:
%cd ~/work/src/
!python train.py mot\
    --data_cfg ./lib/cfg/mot20.json\
    --K 500\
    --exp_id all_hrnet\
    --gpus 0\
    --batch_size 4\
    --arch 'hrnet_18'\
    --num_epochs 60\
    --load_model ../models/fairmot_hrnet_w18.pdparams

# 恢复训练(有bug)

In [ ]:
%cd ~/work/src/
!python train.py mot\
    --data_cfg ./lib/cfg/mot20.json\
    --K 500\
    --exp_id all_hrnet\
    --gpus 0\
    --batch_size 4\
    --arch 'hrnet_18'\
    --num_epochs 60\
    --load_model ../exp/mot/all_hrnet/model_last.pdparams\
    --resume

# 在mot20训练集上eval

In [ ]:
%cd ~/work/src/
!python -W ignore track.py mot\
    --arch 'hrnet_18'\
    --data_dir ../data\
    --val_mot20 True\
    --load_model ../exp/mot/all_hrnet/model_last.pdparams\
    --conf_thres 0.3

# 测试mot20test

In [ ]:
%cd ~/work/src/
!python -W ignore track.py mot\
    --arch 'hrnet_18'\
    --data_dir ../data\
    --test_mot20 True\
    --load_model ../exp/mot/all_hrnet/model_last.pdparams\
    --conf_thres 0.3

# 输入视频，输出追踪结果

In [ ]:
%cd ~/work/src/
!python demo.py mot\
    --load_model ../exp/mot/all_hrnet/model_25.pdparams\
    --arch hrnet_18\
    --reid_dim 128\
    --conf_thres 0.4\
    --input-video ../videos/MOT16-03.mp4\
    --output-root ../videos_results/MOT16-03